In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# Task 1: Read the dataset: okay -_-
file_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(file_path)





In [ ]:
# Task 2: Write your code here: okay -_-
print("--- First 5 rows ---")
display(df.head())

In [ ]:
# Task 3: Write your code here: okay -_-
print("\n--- Dataset Info ---")
df.info()

In [ ]:
# Task 4: Write your code here: okay -_-

print("\n--- Statistical Description ---")
display(df.describe())

In [ ]:
# # Task 1: Write your code here:okay -_-
from sklearn.preprocessing import StandardScaler

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())



In [ ]:
# Task 2: Write your code here:okay -_-
df.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:okay -_-
df = pd.get_dummies(df, drop_first=True)

In [ ]:
if 'prediction' in df.columns:
    target_col = 'prediction'
else:
    target_col = df.columns[-1]

print(f"Detected Target Column: {target_col}")

scaler = StandardScaler()
features_list = [col for col in df.columns if col != target_col]
df[features_list] = scaler.fit_transform(df[features_list])



In [ ]:
# Task 5: Write your code here: okay -_-

print("\nTarget Distribution:\n", df[target_col].value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
import numpy as np

target_col = 'prediction' if 'prediction' in df.columns else df.columns[-1]

X = df.drop(columns=[target_col])
y = df[target_col].astype(int)



In [ ]:
# Task 2,3,4,5: Write your code here:okay -_-
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, silent=True, random_seed=42)

print("Training started...")
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # حساب F1 Score
    score = f1_score(y_test, preds)
    scores.append(score)

full_model_score = np.mean(scores)
print(f"Average F1-Score across all folds: {full_model_score:.4f}")

model.fit(X, y)


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

feature_importance = model.get_feature_importance()
fi_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=fi_df.head(10))
plt.title('Top 10 Features (Identifying the Golden Feature)')
plt.show()


In [ ]:
golden_feature = fi_df.iloc[0]['feature']
print(f"The Golden Feature is: {golden_feature}")

In [ ]:
X_golden = df[[golden_feature]]
golden_scores = []

print(f"Retraining with ONLY the Golden Feature: {golden_feature}")

for train_index, test_index in skf.split(X_golden, y):
    X_train_g, X_test_g = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train_g, y_test_g = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train_g, y_train_g)
    preds_g = model.predict(X_test_g)

    score_g = f1_score(y_test_g, preds_g)
    golden_scores.append(score_g)

# Task 3-4: Print and compare
golden_model_score = np.mean(golden_scores)
print("-" * 30)
print(f"Full Model F1-Score: {full_model_score:.4f}")
print(f"Golden Feature F1-Score: {golden_model_score:.4f}")
print(f"Retained Performance: {(golden_model_score/full_model_score)*100:.2f}%")